In [11]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
from selenium.common.exceptions import TimeoutException


driver = webdriver.Chrome()
search_url ='https://www.daraz.pk/catalog/?q=smartphones'
driver.set_page_load_timeout(30)
try:
    driver.get(search_url)
    time.sleep(2)
    html_content = driver.page_source
    
    
    with open("mobile.html","w",encoding='utf-8') as f:
         f.write(html_content)
         
    print(f"page content save in html file succesffuly")     
    
except TimeoutException:
        print(f'time out check internet connection')   
finally:     
   html_content = driver.page_source
  
driver.quit()       

page content save in html file succesffuly


In [12]:
# check product len
len(html_content)

1053593

In [1]:
from bs4 import BeautifulSoup
with open('data/raw/mobile.html',"r",encoding='utf-8') as f:
    soup= BeautifulSoup(f,'html.parser')
    
product_links = []

for links in soup.find_all("a",href=True):
    href = links["href"]
    
    if href and "/products" in href:
        product_links.append(href)
        
product_links =list(dict.fromkeys(product_links))
    
# Print the total number of unique product links found on the page
print(f'Total Unique Products Links: {len(product_links)}')

Total Unique Products Links: 40


In [4]:
with open('data/raw/mobile.html',"r") as f:
    for i, link in enumerate(product_links, start=1):
        print(i, link)
f.close()

1 //www.daraz.pk/products/apple-iphone-17-pro-max-512gb-i1957775263.html
2 //www.daraz.pk/products/apple-iphone-17-pro-max-256gb-i1958136639.html
3 //www.daraz.pk/products/redmi-15c-4gb-128gb-pta-approved-i923138689.html
4 //www.daraz.pk/products/infinix-smart-20-4gb-64gb-i1958170417.html
5 //www.daraz.pk/products/realme-c100i-4gb64gb-i1958905393.html
6 //www.daraz.pk/products/vivo-y17-pta-approved-original-mobile-phone-8gb-ram-256gb-rom-5000mah-battery-dual-sim-fingerprint-android-smartphone-global-variant-like-new-box-charger-handsfree-included-installment-available-best-budget-phone-in-pakistan-i1958256534.html
7 //www.daraz.pk/products/vivo-y17-635-ips-lcd-8gb-ram-256gb-pta-helio-p35-i491864707.html
8 //www.daraz.pk/products/apple-iphone-16-128gb-i1958148618.html
9 //www.daraz.pk/products/apple-iphone-17-pro-512gb-i1957827008.html
10 //www.daraz.pk/products/realme-gt-7t-12gb-ram-512gb-rom-pta-approved-i895953652.html
11 //www.daraz.pk/products/y85-4-64-622-i1950844162.html
12 //www

In [8]:
# extract data now
import pandas as pd
from selenium import webdriver
import time
import json
from bs4 import BeautifulSoup
import time 
import re

try:
    driver = webdriver.Chrome()
    
    products=[]
    
    for i ,links in enumerate( product_links , start=1):
        if links.startswith('//'):
            links= 'https:'+links
        elif links.startswith('/'):
            links= "https://www.daraz.pk"+links
        if not links.startswith('http'):
            continue
        driver.get(links)
        time.sleep(2)
        
        html_data = driver.page_source
        
        match =  re.search(
        r'var pdpTrackingData = "(.*?)";',
        html_data,
        re.DOTALL
    )
        
        if match:
            try:
               data_string= match.group(1)
               data_string = data_string.replace('\\"', '"')
               
               data=  json.loads(data_string) 
               product = {
                               "Product_Name": data.get("pdt_name"),
                               "Price": data.get("pdt_price"),
                               "Brand": data.get("brand_name"),
                               "SKU": data.get("pdt_sku"),
                               "Category": data.get("pdt_category"),
                               "Seller": data.get("seller_name"),
                               "Product_URL": links
                           }
               products.append(product)
               
               print(f"{i}/40 - {product['Product_Name']}")
               
            except Exception as e:
                           print(f"{i}/40 - JSON Error:", e)
               
        else:
                       print(f"{i}/40 - Tracking data not found")
               
               
    print("\nScraping completed!")
    print("Products collected:", len(products))
except Exception as e:
    print(f'{e}')
    
    

1/40 - Apple iPhone 17 Pro Max - 512GB
2/40 - Apple iPhone 17 Pro Max - 256GB
3/40 - Redmi 15C 4GB +128GB - PTA APPROVED
4/40 - Infinix Smart 20 4GB / 64GB
5/40 - Realme C100i 4GB / 64GB
6/40 - Vivo Y17 PTA Approved Original Mobile Phone | 8GB RAM 256GB ROM | 5000mAh Battery | Dual SIM | Fingerprint | Android Smartphone | Global Variant | Like New | Box Charger Handsfree Included | Installment Available | Best Budget Phone in Pakistan
7/40 - Vivo Y17 - 6.35 Inches IPS LCD Display - 8GB RAM - 256GB Storage - Official PTA Approved - Fingerprint Sensor - Helio P35 Processor - (Phone and Box Charger, No Accessories Included - Mix Color)
8/40 - Apple iPhone 16 - 128GB
9/40 - Apple iPhone 17 Pro - 512GB
10/40 - Realme GT 7T 12GB RAM 512GB ROM - PTA APPROVED
11/40 - Original Vivo Y85 Smartphone | 4GB 64GB | 6.22-inch Notch Screen | PTA Approved | Snapdragon High Performance | FREE Charger Included | Best Discounted Rate | Installment Plan
12/40 - Apple iPhone 17 Pro - 256GB
13/40 - vivo Y05 4

In [13]:
df = pd.DataFrame(products)
len(df)
df

,Product_Name,Price,Brand,SKU,Category,Seller,Product_URL
0,Apple iPhone 17 Pro Max - 512GB,"Rs. 612,799",Apple,1957775263,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-17-...
1,Apple iPhone 17 Pro Max - 256GB,"Rs. 535,600",Apple,1958136639,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-17-...
2,Redmi 15C 4GB +128GB - PTA APPROVED,"Rs. 46,999",Xiaomi Official,923138689,[Mobiles & Tablets],,https://www.daraz.pk/products/redmi-15c-4gb-12...
3,Infinix Smart 20 4GB / 64GB,"Rs. 41,999",Infinix,1958170417,[Mobiles & Tablets],,https://www.daraz.pk/products/infinix-smart-20...
4,Realme C100i 4GB / 64GB,"Rs. 40,999",Realme Official,1958905393,[],,https://www.daraz.pk/products/realme-c100i-4gb...
5,Vivo Y17 PTA Approved Original Mobile Phone | ...,"Rs. 19,799",No Brand,1958256534,[Mobiles & Tablets],,https://www.daraz.pk/products/vivo-y17-pta-app...
6,Vivo Y17 - 6.35 Inches IPS LCD Display - 8GB R...,"Rs. 24,999",No Brand,491864707,[Mobiles & Tablets],,https://www.daraz.pk/products/vivo-y17-635-ips...
7,Apple iPhone 16 - 128GB,"Rs. 345,000",Apple,1958148618,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-16-...
8,Apple iPhone 17 Pro - 512GB,"Rs. 574,100",Apple,1957827008,[Mobiles & Tablets],,https://www.daraz.pk/products/apple-iphone-17-...
9,Realme GT 7T 12GB RAM 512GB ROM - PTA APPROVED,"Rs. 169,999",Realme Official,895953652,[Mobiles & Tablets],,https://www.daraz.pk/products/realme-gt-7t-12g...


In [14]:
df.to_csv('mobile.csv',index=False,encoding='utf-8')

In [5]:
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
import os

driver = webdriver.Chrome()
review_data = []

# Top 10 mobile products ke reviews extract karna
for i, url in enumerate(product_links[30:40], start=1):
  # Protocol fix
  if url.startswith("//"):
    url = "https:" + url
  elif url.startswith("/"):
    url = "https://www.daraz.pk" + url

  if not url.startswith("http"):
    continue

  try:
    driver.get(url)
    time.sleep(3)

    # 1. Step-by-Step Scroll down to trigger Lazy Loading
    for scroll in range(1000, 3000, 700):
      driver.execute_script(f"window.scrollTo(0, {scroll});")
      time.sleep(1)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # Product Name extract
    h1 = soup.find("h1")
    p_name = h1.get_text(strip=True) if h1 else "Mobile Product"

    # 2. Daraz Review Classes Search (Multiple Class Fallbacks)
    comment_blocks = (
        soup.find_all("div", class_="content")
        + soup.find_all("div", class_="item-content")
        + soup.find_all("div", class_="review-content-sl")
    )

    found_count = 0
    seen_texts = set()  # Duplicates se bachne ke liye

    for block in comment_blocks:
      text = block.get_text(strip=True)
      if text and text not in seen_texts and len(text) > 3:
        seen_texts.add(text)
        review_data.append({"Product_Name": p_name, "Review_Text": text})
        found_count += 1

    if found_count > 0:
      print(f"[{i}] Success: Extracted {found_count} reviews for: {p_name[:30]}")
    else:
      print(f"[{i}] No text reviews found for: {p_name[:30]}")

  except Exception as e:
    print(f"[{i}] Error loading page: {e}")

driver.quit()

# CSV Mein Save Karna
file_exists= os.path.exists('Mobile_Reviews.csv')
if review_data:
  df_reviews = pd.DataFrame(review_data)
  df_reviews.to_csv("Mobile_Reviews.csv",mode='a', index=False, encoding="utf-8",header= not file_exists)
  print(
      f"\nMubarak ho! Total {len(df_reviews)} reviews 'Mobile_Reviews.csv' mein"
      " save ho gaye hain."
  )
else:
  print("\nKoi review save nahi ho saka. Mobile links dubara verify karein.")

[1] Error loading page: HTTPConnectionPool(host='localhost', port=57159): Read timed out. (read timeout=120)
[2] Success: Extracted 3 reviews for: OPPO A6s Pro 8GB+256GB
[3] No text reviews found for: vivo V70 FE 8GB+256GB PTA Appr
[4] Success: Extracted 5 reviews for: Oppo Reno 15 5G 256GB
[5] Success: Extracted 1 reviews for: Xiaomi Redmi A7 Pro 4/64
[6] Success: Extracted 7 reviews for: Tecno Camon 50 PRO 8GB/256GB
[7] Success: Extracted 7 reviews for: Samsung Galaxy S26 Ultra - 256
[8] No text reviews found for: Realme GT7 Dream Edition 16GB 
[9] Success: Extracted 7 reviews for: Tecno Camon 50 8GB + 256GB
[10] Success: Extracted 3 reviews for: Oppo A6K 6GB + 128GB

Mubarak ho! Total 33 reviews 'Mobile_Reviews.csv' mein save ho gaye hain.


In [6]:
df = pd.read_csv(r'Mobile_Reviews.csv')
len(df)

177

In [7]:
import os
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver

driver = webdriver.Chrome()
rating_data = []

# Product links par loop (Aap zarurat ke mutabiq [10:20] ya poori list chala sakti hain)
for i, url in enumerate(product_links, start=1):

  # 1. URL Sanitation (Protocol Fixer)
  if url.startswith("//"):
    url = "https:" + url
  elif url.startswith("/"):
    url = "https://www.daraz.pk" + url

  if not url.startswith("http"):
    continue

  try:
    # 2. Page fetch
    driver.get(url)
    time.sleep(3)

    # 3. Scroll Down (Ratings section load hone ke liye)
    for scroll in range(500, 2000, 500):
      driver.execute_script(f"window.scrollTo(0, {scroll});")
      time.sleep(1)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # 4. Product Name extract karna
    h1 = soup.find("h1")
    p_name = h1.get_text(strip=True) if h1 else "Mobile Product"

    # 5. Overall Rating Score (e.g., "4.7")
    score_tag = soup.find("span", class_="score-average")
    overall_rating = score_tag.get_text(strip=True) if score_tag else "N/A"

    # 6. Total Ratings / Review Count (e.g., "120 Ratings")
    count_tag = soup.find("a", class_="pdp-review-summary__link") or soup.find(
        "span", class_="pdp-review-summary__link"
    )
    total_ratings = count_tag.get_text(strip=True) if count_tag else "0"

    # Data collection
    rating_data.append({
        "Product_Name": p_name,
        "Overall_Rating": overall_rating,
        "Total_Ratings": total_ratings,
        "Product_URL": url,
    })

    print(
        f"[{i}] Success: Extracted Rating ({overall_rating}) for:"
        f" {p_name[:30]}"
    )

  except Exception as e:
    print(f"[{i}] Error loading page: {e}")

# Driver Band Karna
driver.quit()

# -------------------------------------------------------------
# CSV Mein Save Karna (Safe Append Mode - Overwrite Nahi Hoga)
# -------------------------------------------------------------
file_name = "Mobile_Ratings.csv"
file_exists = os.path.exists(file_name)

if rating_data:
  df_ratings = pd.DataFrame(rating_data)
  df_ratings.to_csv(
      file_name, mode="a", index=False, header=not file_exists, encoding="utf-8"
  )
  print(
      f"\nMubarak ho! Total {len(df_ratings)} products ki ratings '{file_name}'"
      " mein save ho gayi hain."
  )
else:
  print("\nKoi rating save nahi ho saki.")

[1] Success: Extracted Rating (5.0) for: Apple iPhone 17 Pro Max - 512G
[2] Success: Extracted Rating (4.9) for: Apple iPhone 17 Pro Max - 256G
[3] Success: Extracted Rating (4.7) for: Redmi 15C 4GB +128GB - PTA APP
[4] Success: Extracted Rating (4.9) for: Infinix Smart 20 4GB / 64GB
[5] Success: Extracted Rating (5.0) for: Realme C100i 4GB / 64GB
[6] Success: Extracted Rating (4.0) for: Vivo Y17 PTA Approved Original
[7] Success: Extracted Rating (4.3) for: Vivo Y17 - 6.35 Inches IPS LCD
[8] Success: Extracted Rating (0) for: Apple iPhone 16 - 128GB
[9] Success: Extracted Rating (5.0) for: Apple iPhone 17 Pro - 512GB
[10] Success: Extracted Rating (5.0) for: Realme GT 7T 12GB RAM 512GB RO
[11] Success: Extracted Rating (5.0) for: Original Vivo Y85 Smartphone |
[12] Success: Extracted Rating (N/A) for: Apple iPhone 17 Pro - 256GB
[13] Success: Extracted Rating (N/A) for: vivo Y05 4GB + 64GB PTA Approv
[14] Success: Extracted Rating (4.4) for: Vivo Y66 | Dual Sim | PTA Appr
[15] Success

In [16]:
reviews = pd.read_csv("Mobile_Reviews.csv")
ratings = pd.read_csv("Mobile_Ratings.csv")

merged_df = pd.merge(
    reviews,
    ratings,
    on="Product_Name",
    how="left",
    suffixes=("_Review", "_Rating"),
    sort=False
)

merged_df.to_csv("Mobile_Reviews_Ratings_Merged.csv", index=False, encoding="utf-8")

merged_df

,Product_Name,Review_Text,Overall_Rating,Total_Ratings,Product_URL
0,Apple iPhone 17 Pro Max - 512GB,5.0/52 Ratings20000,5.0,Ratings 2,https://www.daraz.pk/products/apple-iphone-17-...
1,Apple iPhone 17 Pro Max - 512GB,Super Quality with a gift of charger,5.0,Ratings 2,https://www.daraz.pk/products/apple-iphone-17-...
2,Apple iPhone 17 Pro Max - 512GB,Super Quality with a gift of charger0,5.0,Ratings 2,https://www.daraz.pk/products/apple-iphone-17-...
3,Apple iPhone 17 Pro Max - 256GB,4.9/515 Ratings132000,4.9,Ratings 15,https://www.daraz.pk/products/apple-iphone-17-...
4,Apple iPhone 17 Pro Max - 256GB,Great service. All original with seal. Also se...,4.9,Ratings 15,https://www.daraz.pk/products/apple-iphone-17-...
...,...,...,...,...,...
172,Tecno Camon 50 8GB + 256GB,"awesome, I love this, received 2 days late the...",4.2,Ratings 20,https://www.daraz.pk/products/tecno-camon-50-8...
173,Tecno Camon 50 8GB + 256GB,i think i order cammon 50 pro but received cam...,4.2,Ratings 20,https://www.daraz.pk/products/tecno-camon-50-8...
174,Oppo A6K 6GB + 128GB,5.0/51 Ratings10000,5.0,Ratings 1,https://www.daraz.pk/products/oppo-a6k-6gb-128...
175,Oppo A6K 6GB + 128GB,Content:100 Subcategory:100 Depth:100 kamal hai,5.0,Ratings 1,https://www.daraz.pk/products/oppo-a6k-6gb-128...


In [15]:
dfme = pd.read_csv(r'Mobile_Reviews_Ratings_Merged.csv')
len(dfme)

184

In [18]:
mobile_df = pd.read_csv("mobile.csv")
reviews_ratings_df = pd.read_csv("Mobile_Reviews_Ratings_Merged.csv")

final_df = pd.merge(
    mobile_df,
    reviews_ratings_df,
    on="Product_Name",
    how="left",
    suffixes=("_Mobile", "_Reviews_Ratings")
)

final_df.to_csv(
    "Mobile_Final_Merged.csv",
    index=False,
    encoding="utf-8"
)

final_df

,Product_Name,Price,Brand,SKU,Category,Seller,Product_URL_Mobile,Review_Text,Overall_Rating,Total_Ratings,Product_URL_Reviews_Ratings
0,Apple iPhone 17 Pro Max - 512GB,"Rs. 612,799",Apple,1957775263,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/apple-iphone-17-...,5.0/52 Ratings20000,5.0,Ratings 2,https://www.daraz.pk/products/apple-iphone-17-...
1,Apple iPhone 17 Pro Max - 512GB,"Rs. 612,799",Apple,1957775263,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/apple-iphone-17-...,Super Quality with a gift of charger,5.0,Ratings 2,https://www.daraz.pk/products/apple-iphone-17-...
2,Apple iPhone 17 Pro Max - 512GB,"Rs. 612,799",Apple,1957775263,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/apple-iphone-17-...,Super Quality with a gift of charger0,5.0,Ratings 2,https://www.daraz.pk/products/apple-iphone-17-...
3,Apple iPhone 17 Pro Max - 256GB,"Rs. 535,600",Apple,1958136639,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/apple-iphone-17-...,4.9/515 Ratings132000,4.9,Ratings 15,https://www.daraz.pk/products/apple-iphone-17-...
4,Apple iPhone 17 Pro Max - 256GB,"Rs. 535,600",Apple,1958136639,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/apple-iphone-17-...,Great service. All original with seal. Also se...,4.9,Ratings 15,https://www.daraz.pk/products/apple-iphone-17-...
...,...,...,...,...,...,...,...,...,...,...,...
179,Tecno Camon 50 8GB + 256GB,"Rs. 99,999",TECNO,1951494214,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/tecno-camon-50-8...,"awesome, I love this, received 2 days late the...",4.2,Ratings 20,https://www.daraz.pk/products/tecno-camon-50-8...
180,Tecno Camon 50 8GB + 256GB,"Rs. 99,999",TECNO,1951494214,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/tecno-camon-50-8...,i think i order cammon 50 pro but received cam...,4.2,Ratings 20,https://www.daraz.pk/products/tecno-camon-50-8...
181,Oppo A6K 6GB + 128GB,"Rs. 69,999",OPPO Pakistan Official,1950579989,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/oppo-a6k-6gb-128...,5.0/51 Ratings10000,5.0,Ratings 1,https://www.daraz.pk/products/oppo-a6k-6gb-128...
182,Oppo A6K 6GB + 128GB,"Rs. 69,999",OPPO Pakistan Official,1950579989,['Mobiles & Tablets'],NaN,https://www.daraz.pk/products/oppo-a6k-6gb-128...,Content:100 Subcategory:100 Depth:100 kamal hai,5.0,Ratings 1,https://www.daraz.pk/products/oppo-a6k-6gb-128...
